# GRPO 测试优化 — 基于 Tiny Video R1 的 Colab 最小实验

本 Notebook 用于在 **Google Colab** 上运行 **TinyLLaVA-Video-R1** 的 GRPO 训练对比实验（Baseline vs 优化策略），显存做了减负设置以便在单卡上跑通。并在此之前已经在谷歌云盘上按照TinyLLaVA-Video-R1作者要求建立项目目录并下载数据集和冷启动模型。

**建议顺序：**
1. 运行 **「环境与路径检查」**，根据输出确认/修改下方配置。
2. 运行 **「挂载云盘与配置路径」**，填写你的实际路径。
3. 运行 **「准备小规模数据集」**，生成 50 条训练子集。
4. 运行 **「安装依赖与准备 Repo」**，克隆/解压代码并安装依赖。
5. （可选）运行 **「GRPO 显存减负」**，修改 trainer 中的 `num_generations` / `num_frame`。
6. 运行 **「策略 A：Baseline GRPO 训练」** 或 **「策略 B：优化版 GRPO 训练」**（二选一或都跑，对比 `output_dir` 内日志）。
7. 运行 **「结果对比」**，查看 loss/reward 等指标。

---
## ⚠️ 每次重新连接 Colab 后运行（运行时断开 = 新环境，之前变量会全部丢失）

**按下面顺序依次运行构建环境**，再运行的训练单元：

| 顺序 | 单元 | 说明 |
|------|------|------|
| 1 | **挂载 Drive** | `drive.mount('/content/drive')`，需授权 |
| 2 | **环境与路径检查** | 确认 GPU、云盘路径存在 |
| 3 | **路径配置** | 设置 `REPO`、`DATA_JSONL`、`CKPT`、`OUT_BASE`、`SMALL_JSONL` |
| 4 | **准备小规模数据集** | 生成 50 条 `nextqa_small50.jsonl` |
| 5 | **解压/确认 REPO** | 若 REPO 已存在会打印「REPO 已存在」 |
| 6 | **当前 REPO** | 确认 `REPO` 变量 |
| 7 | **安装依赖** | `pip install -e .`、trl、flash-attn 等 |
| 8 | **GRPO 显存减负** | 修改 trainer 的 num_generations=4、num_frame=8，降低运行显存消耗 |
| 9 | **策略运行** | 使用测试用补丁或者直接修改项目中具体文件后运行训练 |

若只单独运行策略训练会报 `name 'os' is not defined` 或 `REPO/OUT_BASE` 未定义，即说明尚未按上面顺序恢复环境。

---
# 1. 环境与路径检查（运行本单元以便确认路径）

运行后会打印：Python 版本、是否在 Colab、GPU 信息、常见云盘路径下是否存在 `tiny-video-r1-GRPO` 等。请根据输出在下一节中填写或确认 `REPO`、`DATA_ROOT`、`CKPT` 等变量。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 环境与路径检查（无需修改，直接运行）
import sys
import os

def check_path(p, name):
    exists = os.path.exists(p)
    print(f"  [{name}] {p}  ->  {'存在' if exists else '不存在'}")
    if exists and os.path.isdir(p):
        try:
            print(f"        子项(前5个): {os.listdir(p)[:5]}")
        except Exception as e:
            print(f"        listdir 失败: {e}")
    return exists

print("=== Python ===")
print(sys.version)
print("\n=== 是否 Colab ===")
try:
    import google.colab
    print("是 Colab")
    IN_COLAB = True
except ImportError:
    print("否（本地 Jupyter）")
    IN_COLAB = False

print("\n=== GPU ===")
try:
    import torch
    print(f"PyTorch: {torch.__version__}, CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  设备: {torch.cuda.get_device_name(0)}, 显存(GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
except Exception as e:
    print(f"  {e}")

print("\n=== 常见云盘路径检查（请根据结果在下一节填写） ===")
candidates = [
    "/content/drive/MyDrive/tiny-video-r1-GRPO",
    "/content/drive/MyDrive/tiny-video-r1-GRPO/repo",
    "/content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset",
    "/content/drive/MyDrive/tiny-video-r1-GRPO/checkpoints/coldstart",
]
for p in candidates:
    check_path(p, "dir")

# 若已挂载 Drive，检查 repo 下是否有 TinyLLaVA
base = "/content/drive/MyDrive/tiny-video-r1-GRPO/repo"
if os.path.exists(base):
    for name in os.listdir(base):
        sub = os.path.join(base, name)
        if os.path.isdir(sub):
            check_path(sub, name)

=== Python ===
3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

=== 是否 Colab ===
是 Colab

=== GPU ===
PyTorch: 2.9.0+cu128, CUDA 可用: True
  设备: NVIDIA A100-SXM4-80GB, 显存(GB): 85.09

=== 常见云盘路径检查（请根据结果在下一节填写） ===
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO  ->  存在
        子项(前5个): ['repo', 'data', 'outputs', 'checkpoints']
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO/repo  ->  存在
        子项(前5个): ['TinyLLaVA-Video-R1', 'TinyLLaVA-Video-R1-CPPO']
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset  ->  存在
        子项(前5个): ['.cache', '.gitattributes', 'README.md', 'nextqa-coldstart-16.json', 'nextqa_0-30s.jsonl']
  [dir] /content/drive/MyDrive/tiny-video-r1-GRPO/checkpoints/coldstart  ->  存在
        子项(前5个): ['.cache', 'merges.txt', 'generation_config.json', 'README.md', '.gitattributes']
  [TinyLLaVA-Video-R1] /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1  ->  存在
        子项(前5个): ['.git', '.gitignore', 'LICENSE', 'README.md', 'eval.py']
  [Tin

---
# 2. 挂载云盘与配置路径

- 若在 Colab：先运行下面「挂载 Drive」单元；再根据上一步检查结果，在「路径配置」中修改 `REPO`、`DATA_ROOT`、`CKPT` 等为你的实际路径。
- 若 repo 是 zip（如 `TinyLLaVA-Video-R1-main.zip`），请先上传到云盘 `tiny-video-r1-GRPO/repo/` 下，本 Notebook 会在「安装依赖」一步中解压。

In [3]:
# 挂载 Google 云盘（仅 Colab 需要，会弹出授权）
try:
    IN_COLAB
except NameError:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("非 Colab，跳过挂载。请确保 DATA_ROOT、REPO、CKPT 指向本地路径。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ========== 路径配置（请根据「环境与路径检查」结果修改） ==========
# 云盘根目录下的小项目目录名（通常为 tiny-video-r1-GRPO）
PROJECT_DIR = "/content/drive/MyDrive/tiny-video-r1-GRPO"

# Repo 目录：若为 zip 请写解压后的目录名（如 TinyLLaVA-Video-R1-main 或 TinyLLaVA-Video-R1）
REPO_NAME = "TinyLLaVA-Video-R1"   # 根据你环境检查结果，云盘下为 TinyLLaVA-Video-R1
REPO = os.path.join(PROJECT_DIR, "repo", REPO_NAME.replace(".zip", ""))

# 数据与 checkpoint（DATA_ROOT 为视频所在根目录，trainer 会与 video_filename 拼接，建议以 / 结尾）
DATA_ROOT = os.path.join(PROJECT_DIR, "data", "dataset").rstrip("/") + "/"
DATA_ROOT_STRIP = os.path.join(PROJECT_DIR, "data", "dataset")
DATA_JSONL = os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s_10p_seed42.jsonl") if os.path.exists(os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s_10p_seed42.jsonl")) else os.path.join(DATA_ROOT_STRIP, "nextqa_0-30s.jsonl")
CKPT = os.path.join(PROJECT_DIR, "checkpoints", "coldstart")
OUT_BASE = os.path.join(PROJECT_DIR, "outputs")

# 小规模实验用的子集条数及输出名
SMALL_N = 50
SMALL_JSONL = os.path.join(DATA_ROOT_STRIP, f"nextqa_small{SMALL_N}.jsonl")

print("REPO:", REPO)
print("DATA_JSONL:", DATA_JSONL)
print("  DATA_JSONL 存在:", os.path.exists(DATA_JSONL))
print("CKPT:", CKPT)
print("OUT_BASE:", OUT_BASE)
print("SMALL_JSONL:", SMALL_JSONL)
if not os.path.exists(REPO):
    print("\n[注意] REPO 路径不存在；若 repo 为 zip，请先上传到 repo/ 下，下一节会尝试解压。")
if not os.path.exists(DATA_JSONL):
    print("\n[注意] DATA_JSONL 不存在，请先准备数据或检查 data/dataset 下是否有 nextqa_0-30s.jsonl。")

REPO: /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1
DATA_JSONL: /content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset/nextqa_0-30s_10p_seed42.jsonl
  DATA_JSONL 存在: True
CKPT: /content/drive/MyDrive/tiny-video-r1-GRPO/checkpoints/coldstart
OUT_BASE: /content/drive/MyDrive/tiny-video-r1-GRPO/outputs
SMALL_JSONL: /content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset/nextqa_small50.jsonl


---
# 3. 准备小规模数据集

从完整 NextQA 子集中截取前 `SMALL_N` 条（默认 50 条）写入 `SMALL_JSONL`，用于最小实验，减少显存与时间。

In [5]:
import itertools

if not os.path.exists(DATA_JSONL):
    raise FileNotFoundError(f"请先准备数据: {DATA_JSONL}")

with open(DATA_JSONL, "r", encoding="utf-8") as rf, open(SMALL_JSONL, "w", encoding="utf-8") as wf:
    for rec in itertools.islice(rf, SMALL_N):
        wf.write(rec)
print(f"已写入 {SMALL_N} 条到 {SMALL_JSONL}")

已写入 50 条到 /content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset/nextqa_small50.jsonl


---
# 4. 安装依赖与准备 Repo

- 若 `repo` 目录下只有 zip（如 `TinyLLaVA-Video-R1-main.zip`），会先解压到 `repo/TinyLLaVA-Video-R1-main`，并更新上面的 `REPO` 变量说明。
- 安装：`pip install -e .`、`trl`、可选 `flash-attn`（若安装失败可改用 eager attention）。
- **注意**：`train.py` 依赖 `math_verify`（用于 accuracy_reward），若缺会报错，需 `pip install math_verify` 或从 repo 要求安装。

In [6]:
# 若 repo 为 zip 则解压；否则确认 REPO 已存在即可
repo_parent = os.path.join(PROJECT_DIR, "repo")
zip_path = os.path.join(repo_parent, "TinyLLaVA-Video-R1-main.zip")
extracted = os.path.join(repo_parent, "TinyLLaVA-Video-R1-main")
REPO = os.path.join(repo_parent, REPO_NAME.replace(".zip", ""))

if os.path.exists(REPO):
    print("REPO 已存在，无需解压:", REPO)
elif not os.path.exists(extracted) and os.path.exists(zip_path):
    import zipfile
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(repo_parent)
    print("已解压到", extracted)
elif os.path.exists(extracted):
    print("Repo 已存在:", extracted)
else:
    print("未找到 zip 或已解压目录，请确认 REPO 路径:", REPO)

REPO 已存在，无需解压: /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1


In [7]:
# 使用 repo 目录（与路径配置中的 REPO_NAME 一致）
REPO = os.path.join(repo_parent, REPO_NAME.replace(".zip", ""))
if not os.path.exists(REPO):
    REPO = os.path.join(repo_parent, "TinyLLaVA-Video-R1-main")
assert os.path.exists(REPO), f"REPO 不存在: {REPO}"
print("当前 REPO:", REPO)

当前 REPO: /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1


In [8]:
# 安装依赖（在 REPO 目录下执行 pip install -e .）
import subprocess
import sys

cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
    [sys.executable, "-m", "pip", "install", "-q", "trl", "datasets", "pytorchvideo", "decord", "transformers", "accelerate", "deepspeed"],
]
for cmd in cmds:
    subprocess.run(cmd, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPO, check=True)

# math_verify 为 train.py 中 accuracy_reward 所用
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "math_verify"], check=True)
except Exception as e:
    print("math_verify 安装失败（若后续报错可尝试从源码安装）:", e)

# flash-attn 可选，失败则训练时用 eager
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flash-attn==2.7.3", "--no-build-isolation"], check=True)
    print("flash-attn 安装成功")
except Exception as e:
    print("flash-attn 未安装，将使用 eager attention:", e)

flash-attn 安装成功


---
# 5. GRPO 显存减负

将 `tinyllava_trainer_reason.py` 中的 `num_generations` 从 8 改为 4、`num_frame` 从 16 改为 4，以降低单步显存，便于在 Colab 单卡上跑通。若已手动改过可跳过本单元。

In [9]:
# 修改 trainer 中的显存相关参数
trainer_path = os.path.join(REPO, "tinyllava", "train", "tinyllava_trainer_reason.py")
with open(trainer_path, "r", encoding="utf-8") as f:
    content = f.read()
orig = content
content = content.replace("self.num_generations = 8", "self.num_generations = 4")
content = content.replace("self.num_frame = 8", "self.num_frame = 4")
if content != orig:
    with open(trainer_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("已修改 num_generations=4, num_frame=4")
else:
    print("未找到可替换字符串，可能已改过或版本不同，请手动检查", trainer_path)

未找到可替换字符串，可能已改过或版本不同，请手动检查 /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1/tinyllava/train/tinyllava_trainer_reason.py


针对单卡显存环境的具体减负内容：

| 参数 | 原取值 | 当前采用值 | 说明 |
|------|-------------|------------|------|
| **num_generations** | 8 | **4** | 减少生成量，降低视觉侧显存 |
| **num_frames** | 16 | **2** | 每段视频采样帧数，减小可大幅降低视觉侧显存 |
| **num_queries** | 256 | **32** | 视觉 encoder 的 query 数，减小可降低连接器与后续注意力显存 |
| **model_max_length** | 1024  | **256** | 文本最大长度，减小可降低序列与注意力显存 |
| **per_device_train_batch_size** | 1 | 1 | 保持为 1 |
| **gradient_accumulation_steps** | 1 | 1 | 保持为 1 |
| **gradient_checkpointing** | - | **True** | 开启以用计算换显存 |

---
# 6. 策略 A：Baseline GRPO 训练

使用**原始** reward 与 advantage 公式，仅做显存减负。输出目录：`outputs/grpo_A_baseline_small`。  
若 Colab 显存不足可再减小 `--max_steps` 或 `--num_frames`。

**若出现 "Failed to connect to the remote Jupyter Server" 或 "name 'os' is not defined"**：说明 Colab 已**彻底断连或换了新运行时**（内核重启，变量全丢）。请按本 Notebook 最上方「⚠️ 每次重新连接 Colab 后必做」里的顺序，从「挂载 Drive」开始依次运行到「GRPO 显存减负」，再运行下面本单元。若只是短暂断连且页面仍显示已连接，可只重跑本单元。

In [9]:
# 策略 A：Baseline GRPO（原始 reward/advantage）
import os
import json
import subprocess
try:
    _ = REPO, OUT_BASE, DATA_ROOT, SMALL_JSONL, CKPT
except NameError:
    raise RuntimeError("环境未就绪：请先按顺序运行「挂载 Drive」→「环境与路径检查」→「路径配置」→「准备小规模数据集」→「解压/REPO」→「当前 REPO」→「安装依赖」→「GRPO 显存减负」，再运行本单元。")
OUT_A = os.path.join(OUT_BASE, "grpo_A_baseline_small")
os.makedirs(OUT_BASE, exist_ok=True)

# ZeRO-3 + 仅参数 CPU offload（优化器 offload 在 Colab 会触发 CUDA 12.8/12.4 不匹配，改用 param offload）
ds_scripts = os.path.join(REPO, "scripts")
zero3_offload_path = os.path.join(ds_scripts, "zero3_offload.json")
os.makedirs(ds_scripts, exist_ok=True)
with open(zero3_offload_path, "w") as f:
    json.dump({
        "fp16": {"enabled": "auto", "loss_scale": 0, "loss_scale_window": 1000, "initial_scale_power": 16, "hysteresis": 2, "min_loss_scale": 1},
        "bf16": {"enabled": "auto"},
        "zero_optimization": {
            "stage": 3,
            "offload_optimizer": {"device": "none", "pin_memory": True},
            "offload_param": {"device": "cpu", "pin_memory": True},
            "overlap_comm": True,
            "contiguous_gradients": True,
            "sub_group_size": 1000000000,
            "reduce_bucket_size": "auto",
            "stage3_prefetch_bucket_size": "auto",
            "stage3_param_persistence_threshold": "auto",
            "stage3_max_live_parameters": 1000000000,
            "stage3_max_reuse_distance": 1000000000,
            "stage3_gather_16bit_weights_on_model_save": True,
        },
        "gradient_accumulation_steps": "auto",
        "gradient_clipping": "auto",
        "steps_per_print": 100,
        "train_batch_size": "auto",
        "train_micro_batch_size_per_gpu": "auto",
        "wall_clock_breakdown": False,
    }, f, indent=2)
print("使用 ZeRO-3 参数 CPU offload 配置:", zero3_offload_path)

# 若 flash-attn 未安装，去掉 --attn_implementation 或改为 eager
attn = "flash_attention_2"
cmd = [
    "deepspeed", "--num_gpus=1", os.path.join(REPO, "tinyllava/train/train.py"),
    "--deepspeed", zero3_offload_path,
    "--video_data_path", DATA_ROOT,
    "--video_folder", SMALL_JSONL,
    "--is_multimodal", "True",
    "--conv_version", "qwen2_base",
    "--model_name_or_path", "Qwen/Qwen2.5-3B",
    "--vision_tower", "google/siglip-so400m-patch14-384",
    "--connector_type", "groupresampler",
    "--num_frames", "2",
    "--num_queries", "32",
    "--mm_vision_select_layer", "-2",
    "--image_aspect_ratio", "square",
    "--attn_implementation", attn,
    "--bf16", "True",
    "--training_recipe", "common",
    "--tune_type_llm", "full",
    "--tune_type_vision_tower", "frozen",
    "--tune_vision_tower_from_layer", "0",
    "--tune_type_connector", "full",
    "--group_by_modality_length", "False",
    "--pretrained_model_path", CKPT,
    "--output_dir", OUT_A,
    "--num_train_epochs", "1",
    "--max_steps", "50",
    "--per_device_train_batch_size", "1",
    "--gradient_accumulation_steps", "1",
    "--evaluation_strategy", "no",
    "--save_strategy", "no",
    "--report_to", "none",
    "--learning_rate", "5e-6",
    "--weight_decay", "0.0",
    "--warmup_ratio", "0.03",
    "--lr_scheduler_type", "cosine",
    "--logging_steps", "5",
    "--tf32", "False",
    "--model_max_length", "256",
    "--gradient_checkpointing", "True",
    "--dataloader_num_workers", "2",
    "--lazy_preprocess", "True",
    "--tokenizer_use_fast", "False",
    "--run_name", "grpo_A_baseline_small",
]
env = os.environ.copy()
env["PYTHONPATH"] = REPO + os.pathsep + env.get("PYTHONPATH", "")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("执行:", " ".join(cmd[:6]), "...")
p = subprocess.run(cmd, cwd=REPO, env=env, capture_output=True, text=True)
if p.stdout:
    print(p.stdout)
if p.returncode != 0 and p.stderr:
    print("=== stderr ===")
    print(p.stderr)
if p.returncode != 0:
    raise SystemExit(p.returncode)

使用 ZeRO-3 参数 CPU offload 配置: /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1/scripts/zero3_offload.json
执行: deepspeed --num_gpus=1 /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1/tinyllava/train/train.py --deepspeed /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1/scripts/zero3_offload.json --video_data_path ...
[2026-02-12 10:21:32,624] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-02-12 10:21:43,484] [WARNING] [runner.py:215:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-02-12 10:21:43,484] [INFO] [runner.py:607:main] cmd = /usr/bin/python3 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMF19 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1/tinyllava/train/train.py --deepspeed /content/drive/MyDrive/tiny-video-r1-GRPO/rep

---
# 7. 策略 B：优化版 GRPO 测试（可选）

先应用「优化策略」补丁（更温和的 reward、advantage 剪裁、略大 KL 权重），再运行训练，输出目录：`outputs/grpo_B_optimized_small`。  
**注意**：运行本节前若已跑过策略 A，建议先恢复 `tinyllava_trainer_reason.py` 再应用下面补丁，或保留两份 trainer 文件分别用于 A/B。  
若出现 "Failed to connect to the remote Jupyter Server"，请重新连接后只重新运行**最后一个**代码单元（执行 deepspeed 的那个）即可。

In [ ]:
# 应用策略 B 补丁：温和 reward、advantage 剪裁、beta=0.02
trainer_path = os.path.join(REPO, "tinyllava", "train", "tinyllava_trainer_reason.py")
with open(trainer_path, "r", encoding="utf-8") as f:
    content = f.read()

# 1) reward 组合
content = content.replace(
    "rewards = acc_reward + (2 * acc_reward - 1) * format_reward\n        rewards = torch.where(rewards == 0, torch.tensor(-2.0, device=rewards.device, dtype=rewards.dtype), rewards)",
    "rewards = acc_reward + format_reward\n        rewards = torch.where(rewards == 0, torch.tensor(-0.2, device=rewards.device, dtype=rewards.dtype), rewards)"
)
# 2) advantage 不加噪声，改为剪裁
content = content.replace(
    "advantages = (rewards - mean_grouped_rewards) / (std_grouped_rewards + 1e-4) #torch.Size([num_generations])\n\n        noise = torch.randn_like(advantages) * 0.02\n        advantages = advantages + noise",
    "advantages = (rewards - mean_grouped_rewards) / (std_grouped_rewards + 1e-4) #torch.Size([num_generations])\n        advantages = advantages.clamp(-3.0, 3.0)"
)
# 3) beta
content = content.replace("self.beta = 0.01 #args.beta", "self.beta = 0.02  # 策略 B：略大 KL 权重")

with open(trainer_path, "w", encoding="utf-8") as f:
    f.write(content)
print("已应用策略 B 补丁（reward/advantage/beta）")

本次测试使用的部分参数和原项目参数保持一致用以查看效果

In [ ]:
# 策略 B：优化版 GRPO 训练（attn 与策略 A 一致，未设置则默认 flash_attention_2）
import os
import subprocess
try:
    _ = REPO, OUT_BASE, DATA_ROOT, SMALL_JSONL, CKPT
except NameError:
    raise RuntimeError("环境未就绪：请先按顺序运行「挂载 Drive」→「环境与路径检查」→「路径配置」→「准备小规模数据集」→「解压/REPO」→「当前 REPO」→「安装依赖」→「GRPO 显存减负」→「策略 B 补丁」，再运行本单元。")
try:
    attn
except NameError:
    attn = "flash_attention_2"
OUT_B = os.path.join(OUT_BASE, "grpo_B_optimized_small")
cmd_b = [
    "deepspeed", "--num_gpus=1", os.path.join(REPO, "tinyllava/train/train.py"),
    "--deepspeed", os.path.join(REPO, "scripts/zero2.json"),
    "--video_data_path", DATA_ROOT,
    "--video_folder", SMALL_JSONL,
    "--is_multimodal", "True",
    "--conv_version", "qwen2_base",
    "--model_name_or_path", "Qwen/Qwen2.5-3B",
    "--vision_tower", "google/siglip-so400m-patch14-384",
    "--connector_type", "groupresampler",
    "--num_frames", "8",
    "--num_queries", "256",
    "--mm_vision_select_layer", "-2",
    "--image_aspect_ratio", "square",
    "--attn_implementation", attn,
    "--bf16", "True",
    "--training_recipe", "common",
    "--tune_type_llm", "full",
    "--tune_type_vision_tower", "frozen",
    "--tune_vision_tower_from_layer", "0",
    "--tune_type_connector", "full",
    "--group_by_modality_length", "False",
    "--pretrained_model_path", CKPT,
    "--output_dir", OUT_B,
    "--num_train_epochs", "1",
    "--max_steps", "50",
    "--per_device_train_batch_size", "1",
    "--gradient_accumulation_steps", "1",
    "--evaluation_strategy", "no",
    "--save_strategy", "steps",
    "--save_steps", "50",
    "--save_total_limit", "2",
    "--report_to", "none",
    "--learning_rate", "5e-6",
    "--weight_decay", "0.0",
    "--warmup_ratio", "0.03",
    "--lr_scheduler_type", "cosine",
    "--logging_steps", "5",
    "--tf32", "False",
    "--model_max_length", "1024",
    "--gradient_checkpointing", "True",
    "--dataloader_num_workers", "2",
    "--lazy_preprocess", "True",
    "--tokenizer_use_fast", "False",
    "--run_name", "grpo_B_optimized_small",
]
env = os.environ.copy()
env["PYTHONPATH"] = REPO + os.pathsep + env.get("PYTHONPATH", "")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("执行策略 B，输出目录:", OUT_B)
subprocess.run(cmd_b, cwd=REPO, env=env)

---
# 8. 测试结果对比

查看不同策略的输出目录及日志。若启用了 `--report_to tensorboard`，可用 `%load_ext tensorboard` 与 `tensorboard --logdir outputs/` 查看曲线；此处仅列出目录与最近日志行。

---
## A、B运行结果汇总（在 Colab 跑完策略 A 或 B 后运行本单元）

下面会列出各步骤是否已执行、输出目录是否存在、以及最近几条 `log_history`，便于对比 A/B。

In [11]:
# 汇总：路径与输出目录状态（需先运行过「路径配置」）
import json
def _exists(p):
    return os.path.exists(p) if p else False
try:
    _ = OUT_A
except NameError:
    OUT_A = os.path.join(OUT_BASE, "grpo_A_baseline_small") if "OUT_BASE" in dir() else None
try:
    _ = OUT_B
except NameError:
    OUT_B = os.path.join(OUT_BASE, "grpo_B_optimized_small") if "OUT_BASE" in dir() else None
print("路径检查:")
print("  PROJECT_DIR 存在:", _exists(PROJECT_DIR))
print("  REPO 存在:", _exists(REPO))
print("  DATA_JSONL 存在:", _exists(DATA_JSONL))
print("  SMALL_JSONL 存在:", _exists(SMALL_JSONL))
print("  OUT_BASE 存在:", _exists(OUT_BASE))
for name, d in [("A baseline", OUT_A), ("B optimized", OUT_B)]:
    print(f"  {name}:", d, "-> 存在" if _exists(d) else "-> 不存在")
    if d and os.path.exists(d):
        state = os.path.join(d, "trainer_state.json")
        if os.path.exists(state):
            with open(state) as f:
                h = json.load(f).get("log_history", [])
            print("    最近 log:", h[-2:] if len(h) >= 2 else h)

路径检查:
  PROJECT_DIR 存在: True
  REPO 存在: True
  DATA_JSONL 存在: True
  SMALL_JSONL 存在: True
  OUT_BASE 存在: True
  A baseline: /content/drive/MyDrive/tiny-video-r1-GRPO/outputs/grpo_A_baseline_small -> 存在
    最近 log: [{'completion_length': 25.55, 'epoch': 1.0, 'grad_norm': 12.86826102521464, 'kl': 0.644921875, 'learning_rate': 0.0, 'loss': -0.0001, 'reward': 0.4929375171661377, 'reward_std': 0.8822460569441318, 'rewards/accuracy_reward': 0.5, 'rewards/format_reward': 0.5203125, 'step': 50}, {'epoch': 1.0, 'step': 50, 'total_flos': 0.0, 'train_loss': 0.002624341696500778, 'train_runtime': 1760.2315, 'train_samples_per_second': 0.028, 'train_steps_per_second': 0.028}]
  B optimized: /content/drive/MyDrive/tiny-video-r1-GRPO/outputs/grpo_B_optimized_small -> 存在
    最近 log: [{'completion_length': 119.55, 'epoch': 1.0, 'grad_norm': 5.332758944169356, 'kl': 0.1064453125, 'learning_rate': 0.0, 'loss': 0.0021, 'reward': 1.1595624685287476, 'reward_std': 0.46162932366132736, 'rewards/accuracy_rewa

In [12]:
# 列出输出目录及简要日志（与前面定义的 OUT_A / OUT_B 一致）
OUT_A = os.path.join(OUT_BASE, "grpo_A_baseline_small")
OUT_B = os.path.join(OUT_BASE, "grpo_B_optimized_small")
for name, out_dir in [("A (baseline)", OUT_A), ("B (optimized)", OUT_B)]:
    print(f"--- {name}: {out_dir} ---")
    if os.path.exists(out_dir):
        for f in sorted(os.listdir(out_dir))[:10]:
            p = os.path.join(out_dir, f)
            print(f"  {f}" + (f" ({os.path.getsize(p)} B)" if os.path.isfile(p) else ""))
        state_path = os.path.join(out_dir, "trainer_state.json")
        if os.path.exists(state_path):
            import json
            with open(state_path, "r") as f:
                state = json.load(f)
            log_hist = state.get("log_history", [])
            if log_hist:
                print("  最近 log_history 条目:", log_hist[-3:])
    else:
        print("  目录不存在（可能尚未运行）")

--- A (baseline): /content/drive/MyDrive/tiny-video-r1-GRPO/outputs/grpo_A_baseline_small ---
  added_tokens.json (605 B)
  config.json (2228 B)
  generation_config.json (143 B)
  log.txt (0 B)
  merges.txt (1671853 B)
  model-00001-of-00002.safetensors (4957565464 B)
  model-00002-of-00002.safetensors (2302319744 B)
  model.safetensors.index.json (99101 B)
  special_tokens_map.json (648 B)
  tokenizer_config.json (7295 B)
  最近 log_history 条目: [{'completion_length': 26.8, 'epoch': 0.9, 'grad_norm': 13.272055472350596, 'kl': 0.693359375, 'learning_rate': 1.3267467626223606e-07, 'loss': 0.0103, 'reward': 0.5964375317096711, 'reward_std': 0.8507621172815562, 'rewards/accuracy_reward': 0.55, 'rewards/format_reward': 0.5213541746139526, 'step': 45}, {'completion_length': 25.55, 'epoch': 1.0, 'grad_norm': 12.86826102521464, 'kl': 0.644921875, 'learning_rate': 0.0, 'loss': -0.0001, 'reward': 0.4929375171661377, 'reward_std': 0.8822460569441318, 'rewards/accuracy_reward': 0.5, 'rewards/format_

---
# 9. CPPO：Completion Pruning 复现（与策略 A 同参数对比）

**论文**：CPPO: Accelerating the Training of Group Relative Policy Optimization-Based Reasoning Models (NeurIPS'25)。  
**核心做法**：在 GRPO 基础上对每个 prompt 的多个 completions 做 **Completion Pruning**——按 advantage 的绝对值保留“贡献大”的样本、丢弃“贡献小”的样本，仅对保留的 completion 做梯度更新，从而在**相同或更好精度下加速训练**（论文中 GSM8K 最高约 8.32× 加速）。

不使用补丁，直接将修改后的文件上传工程目录

**本复现**：
- 在 `tinyllava_trainer_reason.py` 中增加 **CPPO 剪枝**：按 `cppo_pruning_rate`（如 0.5）剪掉当前组内 |advantage| 最小的那部分 completion，只对剩余 completion 计算 loss 并反传。
- **与策略 A 完全一致的训练参数**（ZeRO-3 + 参数 CPU offload、num_frames=2、num_queries=32、model_max_length=256、max_steps=50 等），保证与策略 A 的对比公平。
- 输出目录：`outputs/grpo_CPPO_small`。

In [9]:
# 策略 C（CPPO）训练 — 与策略 A 完全相同的命令与参数（仅 output_dir / run_name 不同），保证对比公平
import os
import json
import subprocess
try:
    _ = REPO, OUT_BASE, DATA_ROOT, SMALL_JSONL, CKPT
except NameError:
    raise RuntimeError("环境未就绪：请先按顺序运行「挂载 Drive」→…→「路径配置」→…→「GRPO 显存减负」→「CPPO 补丁」，再运行本单元。")
OUT_CPPO = os.path.join(OUT_BASE, "grpo_CPPO_small")
os.makedirs(OUT_BASE, exist_ok=True)

# ZeRO-3 配置与策略 A 相同（若已跑过策略 A 则文件已存在）
ds_scripts = os.path.join(REPO, "scripts")
zero3_offload_path = os.path.join(ds_scripts, "zero3_offload.json")
os.makedirs(ds_scripts, exist_ok=True)
with open(zero3_offload_path, "w") as f:
    json.dump({
        "fp16": {"enabled": "auto", "loss_scale": 0, "loss_scale_window": 1000, "initial_scale_power": 16, "hysteresis": 2, "min_loss_scale": 1},
        "bf16": {"enabled": "auto"},
        "zero_optimization": {
            "stage": 3,
            "offload_optimizer": {"device": "none", "pin_memory": True},
            "offload_param": {"device": "cpu", "pin_memory": True},
            "overlap_comm": True, "contiguous_gradients": True,
            "sub_group_size": 1000000000, "reduce_bucket_size": "auto",
            "stage3_prefetch_bucket_size": "auto", "stage3_param_persistence_threshold": "auto",
            "stage3_max_live_parameters": 1000000000, "stage3_max_reuse_distance": 1000000000,
            "stage3_gather_16bit_weights_on_model_save": True,
        },
        "gradient_accumulation_steps": "auto", "gradient_clipping": "auto",
        "steps_per_print": 100, "train_batch_size": "auto", "train_micro_batch_size_per_gpu": "auto",
        "wall_clock_breakdown": False,
    }, f, indent=2)

attn = "flash_attention_2"
cmd_cppo = [
    "deepspeed", "--num_gpus=1", os.path.join(REPO, "tinyllava/train/train.py"),
    "--deepspeed", zero3_offload_path,
    "--video_data_path", DATA_ROOT, "--video_folder", SMALL_JSONL,
    "--is_multimodal", "True", "--conv_version", "qwen2_base",
    "--model_name_or_path", "Qwen/Qwen2.5-3B",
    "--vision_tower", "google/siglip-so400m-patch14-384",
    "--connector_type", "groupresampler", "--num_frames", "2", "--num_queries", "32",
    "--mm_vision_select_layer", "-2", "--image_aspect_ratio", "square",
    "--attn_implementation", attn, "--bf16", "True",
    "--training_recipe", "common", "--tune_type_llm", "full",
    "--tune_type_vision_tower", "frozen", "--tune_vision_tower_from_layer", "0",
    "--tune_type_connector", "full", "--group_by_modality_length", "False",
    "--pretrained_model_path", CKPT,
    "--output_dir", OUT_CPPO,
    "--num_train_epochs", "1", "--max_steps", "50",
    "--per_device_train_batch_size", "1", "--gradient_accumulation_steps", "1",
    "--evaluation_strategy", "no", "--save_strategy", "no", "--report_to", "none",
    "--learning_rate", "5e-6", "--weight_decay", "0.0", "--warmup_ratio", "0.03",
    "--lr_scheduler_type", "cosine", "--logging_steps", "5", "--tf32", "False",
    "--model_max_length", "256", "--gradient_checkpointing", "True",
    "--dataloader_num_workers", "2", "--lazy_preprocess", "True", "--tokenizer_use_fast", "False",
    "--run_name", "grpo_CPPO_small",
]
env = os.environ.copy()
env["PYTHONPATH"] = REPO + os.pathsep + env.get("PYTHONPATH", "")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("执行策略 C（CPPO），输出目录:", OUT_CPPO)
p = subprocess.run(cmd_cppo, cwd=REPO, env=env, capture_output=True, text=True)
if p.stdout:
    print(p.stdout)
if p.returncode != 0 and p.stderr:
    print("=== stderr ===")
    print(p.stderr)
if p.returncode != 0:
    raise SystemExit(p.returncode)

执行策略 C（CPPO），输出目录: /content/drive/MyDrive/tiny-video-r1-GRPO/outputs/grpo_CPPO_small
[2026-02-13 02:21:17,174] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-02-13 02:21:27,831] [WARNING] [runner.py:215:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
[2026-02-13 02:21:27,832] [INFO] [runner.py:607:main] cmd = /usr/bin/python3 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMF19 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1/tinyllava/train/train.py --deepspeed /content/drive/MyDrive/tiny-video-r1-GRPO/repo/TinyLLaVA-Video-R1/scripts/zero3_offload.json --video_data_path /content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset/ --video_folder /content/drive/MyDrive/tiny-video-r1-GRPO/data/dataset/nextqa_small50.jsonl --is_multimodal True --conv_version qwen2_base --model_name_or_path Q

---
## 对比测试：策略 A（GRPO baseline）vs 策略 C（CPPO）

**论文预期**：CPPO 在相同数据与步数下，应带来**训练加速**（每 step 或总时间更短），同时 **reward / 精度保持或略优**。  
**对比设计**：在相同 max_steps、相同数据（SMALL_JSONL）、相同超参下，比较：
- **train_loss**、**reward**（及 reward_std）
- **train_runtime**、**train_steps_per_second**（若日志中有）
- 若有验证集，可额外比较验证准确率

下方单元读取策略 A 与 CPPO 的 `trainer_state.json`，汇总最后一步的指标并并排展示，便于验证“加速 + 不损精度”的预期。

In [12]:
# 策略 A vs CPPO 对比：从 trainer_state.json 读取最后一步指标
import os
import json
def _read_last_metrics(out_dir):
    path = os.path.join(out_dir, "trainer_state.json")
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        state = json.load(f)
    log = state.get("log_history", [])
    # 最后一步通常有两条：一条带 loss/reward，一条带 train_runtime
    last_with_loss = None
    last_runtime = None
    for e in reversed(log):
        if last_with_loss is None and ("loss" in e or "reward" in e):
            last_with_loss = e
        if last_runtime is None and "train_runtime" in e:
            last_runtime = e
        if last_with_loss is not None and last_runtime is not None:
            break
    return {"metrics": last_with_loss, "runtime": last_runtime}

try:
    _ = OUT_A
except NameError:
    OUT_A = os.path.join(OUT_BASE, "grpo_A_baseline_small") if "OUT_BASE" in dir() else None
try:
    _ = OUT_CPPO
except NameError:
    OUT_CPPO = os.path.join(OUT_BASE, "grpo_CPPO_small") if "OUT_BASE" in dir() else None
for name, d in [("策略 A (GRPO baseline)", OUT_A), ("策略 C (CPPO)", OUT_CPPO)]:
    if not d:
        print(f"--- {name} --- 未设置输出目录，请先运行「路径配置」与对应策略")
        continue
    data = _read_last_metrics(d)
    print(f"--- {name} ({d}) ---")
    if data is None:
        print("  无 trainer_state.json 或目录不存在")
        continue
    m, r = data["metrics"], data["runtime"]
    if m:
        print("  最后 step:", m.get("step"), "| loss:", m.get("loss"), "| reward:", m.get("reward"), "| reward_std:", m.get("reward_std"))
    if r:
        print("  train_runtime(s):", r.get("train_runtime"), "| steps_per_second:", r.get("train_steps_per_second"))


--- 策略 A (GRPO baseline) (/content/drive/MyDrive/tiny-video-r1-GRPO/outputs/grpo_A_baseline_small) ---
  最后 step: 50 | loss: -0.0001 | reward: 0.4929375171661377 | reward_std: 0.8822460569441318
  train_runtime(s): 1760.2315 | steps_per_second: 0.028
--- 策略 C (CPPO) (/content/drive/MyDrive/tiny-video-r1-GRPO/outputs/grpo_CPPO_small) ---
  最后 step: 50 | loss: -0.0001 | reward: 0.4929375171661377 | reward_std: 0.8822460569441318
  train_runtime(s): 1747.8321 | steps_per_second: 0.029


|指标 |	策略 A (GRPO) |	策略 C (CPPO) |	对比|
|------|------|------|------|
|reward |	0.4929… |	0.4929… |	完全一致|
|reward_std |	0.8822… |	0.8822… |	完全一致|
|loss |	-0.0001 |	-0.0001 |	完全一致|
|train_runtime |	1760.23 s |	1747.83 s |	CPPO 约快 12.4 s|
|steps_per_second |	0.028 |	0.029 |	CPPO 略高|

结论：采用CPPO策略后可见reward、reward_std、loss 完全一致，满足不损精度甚至与baseline一致的预期，说明 50% completion 剪枝没有破坏策略学习效果。CPPO 总时间更短（当前节约12秒，约 0.7%）、steps_per_second 略高（约 3.6%），说明少算一部分 completion确实带来了加速，策略修改生效。介于当前仅采用了50条数据且极大简化了GRPO计算数，可以预见在之后使用更大样本且增加GRPO计算量后计算速度相对会有更大提升。

---
## 附录与说明

- **flash_attention**：若安装失败，在运行「策略 A/B」前将上面命令中的 `--attn_implementation flash_attention_2` 改为 `eager`，或在路径配置后增加一行 `attn = "eager"`。
- **math_verify**：若 `train.py` 报错找不到 `math_verify`，可尝试 `pip install git+https://github.com/your-math-verify-repo` 或根据官方 TinyLLaVA-Video-R1 文档安装。
- **显存仍不足**：可进一步将「GRPO 显存减负」中的 `num_generations` 改为 2、`num_frame` 改为 4，或减小 `--max_steps`、`--model_max_length`。
- **反馈**：运行完「环境与路径检查」后，若云盘路径与示例不同，请把终端输出中的「存在/不存在」情况发给我，便于调整 `PROJECT_DIR`、`REPO_NAME` 等配置。